[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# Taller 10 de  Física Computacional
Solución de Taller 10 del taller de física computacional.

**Estudiante:** Erick Alexander Muñoz Orozco

**Lenguaje:** Fortran 2008 y Python

**Tema:** Librería de integración en Fortran, usada desde Python



---



In [25]:
# Comprueba gfortran (y lo instala en Colab si falta) y crea la carpeta del Taller
!which gfortran > /dev/null || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
!gfortran --version | head -1
!mkdir -p taller_cuerpo_negro
import os
os.makedirs("/content/taller_cuerpo_negro", exist_ok=True)
%cd /content/taller_cuerpo_negro

GNU Fortran (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
/content/taller_cuerpo_negro


**Parte B. Parte Fortran: La librería**

In [44]:
!pip install -q meson ninja

In [45]:
# Carpeta de trabajo del taller: aquí se escriben, compilan, importan y verifican los archivos
import os, sys
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:     # evita crear taller_f2py/taller_f2py
    os.makedirs(CARPETA, exist_ok=True)
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())               # import integ busca la librería aquí
print("Carpeta de trabajo:", os.getcwd())

Carpeta de trabajo: /content/taller_cuerpo_negro/taller_f2py


Modulo

In [46]:
%%writefile modulo_integracion.f90
! ============================================================
! Módulo:    integracion
! Propósito: reglas compuestas del trapecio y de Simpson 1/3
!            para ser compiladas con F2PY.
! ============================================================
module integracion
  implicit none
contains

  ! ----------------------------------------------------------
  ! trapecio: regla del trapecio compuesta con paso uniforme
  ! Entradas: n, número de puntos de la malla
  !           y(n), valores de la función en los puntos
  !           h, paso uniforme de la malla
  ! Salida:   s, aproximación de la integral
  ! ----------------------------------------------------------
  subroutine trapecio(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s

    if (n < 2) then
      s = 0.0d0
      return
    end if

    s = h * (0.5d0 * (y(1) + y(n)) + sum(y(2:n-1)))
  end subroutine trapecio

  ! ----------------------------------------------------------
  ! simpson: regla de Simpson 1/3 compuesta (requiere n impar)
  ! Entradas: n, número impar de puntos de la malla
  !           y(n), valores de la función en los puntos
  !           h, paso uniforme de la malla
  ! Salida:   s, aproximación de la integral
  ! ----------------------------------------------------------
  subroutine simpson(n, y, h, s)
    integer, intent(in)  :: n
    real(8), intent(in)  :: y(n), h
    real(8), intent(out) :: s

    ! Si n < 3 o es par (mod(n, 2) == 0), no aplica Simpson 1/3;
    ! devolvemos un indicador sin usar STOP para no tumbar Python.
    if (n < 3 .or. mod(n, 2) == 0) then
      s = -1.0d30
      return
    end if

    s = (h / 3.0d0) * (y(1) + y(n) + 4.0d0 * sum(y(2:n-1:2)) + 2.0d0 * sum(y(3:n-2:2)))
  end subroutine simpson

end module integracion

Overwriting modulo_integracion.f90


In [47]:
# Compila la librería "integ" (ejecute esta celda cuando su archivo esté listo)
!python -m numpy.f2py -c modulo_integracion.f90 -m integ 2>&1 | tail -n 3

              s = simpson(y,h,[n])
    Wrote C/API module "integ" to file "./integmodule.c"
    Fortran 90 wrappers are saved to "./integ-f2pywrappers2.f90"


**Parte C. Parte Python: uso de la librería**

Actividad 1 — Firma de la librería

In [48]:
import integ

# Acceso compatible tanto si F2PY expuso el submodulo 'integracion' como si esta en el nivel raiz
mod = integ.integracion if hasattr(integ, "integracion") else integ

print(mod.trapecio.__doc__)

s = trapecio(y,h,[n])

Wrapper for ``trapecio``.

Parameters
----------
y : input rank-1 array('d') with bounds (n)
h : input float

Other Parameters
----------------
n : input int, optional
    Default: shape(y, 0)

Returns
-------
s : float



Actividad 2 — Orden de convergencia

In [49]:
import numpy as np

ns = [11, 21, 41, 81]
I_exacto = 2.0

err_trap = []
err_simp = []

print(f"{'n':>4} | {'h':>10} | {'Err Trapecio':>14} | {'Razón T':>9} | {'Err Simpson':>14} | {'Razón S':>9}")
print("-" * 73)

for i, n in enumerate(ns):
    x = np.linspace(0.0, np.pi, n)
    h = x[1] - x[0]
    y = np.sin(x)

    s_t = mod.trapecio(y, h)
    s_s = mod.simpson(y, h)

    et = abs(s_t - I_exacto)
    es = abs(s_s - I_exacto)
    err_trap.append(et)
    err_simp.append(es)

    razon_t = f"{err_trap[i-1]/et:9.4f}" if i > 0 else f"{'-':>9}"
    razon_s = f"{err_simp[i-1]/es:9.4f}" if i > 0 else f"{'-':>9}"

    print(f"{n:4d} | {h:10.6f} | {et:14.6e} | {razon_t} | {es:14.6e} | {razon_s}")

   n |          h |   Err Trapecio |   Razón T |    Err Simpson |   Razón S
-------------------------------------------------------------------------
  11 |   0.314159 |   1.647646e-02 |         - |   1.095173e-04 |         -
  21 |   0.157080 |   4.114027e-03 |    4.0049 |   6.784442e-06 |   16.1424
  41 |   0.078540 |   1.028190e-03 |    4.0012 |   4.230932e-07 |   16.0353
  81 |   0.039270 |   2.570276e-04 |    4.0003 |   2.642876e-08 |   16.0088


Actividad 3 — Constante de Stefan-Boltzmann

In [41]:
import numpy as np

# Constantes físicas CODATA (SI)
k_B = 1.380649e-23      # J / K
h_planck = 6.62607015e-34  # J * s
c = 299792458.0         # m / s

n = 2001
t = np.linspace(0.0, 1.0, n)
ht = t[1] - t[0]

# Evaluamos solo en el interior (0 < t < 1) para evitar indeterminación 0/0 y overflow en t=1
f_t = np.zeros(n, dtype=np.float64)
t_int = t[1:-1]
x_int = t_int / (1.0 - t_int)

# Protección contra overflow de exp(x) cuando t -> 1 (x > 700)
mask = x_int < 700.0
xm = x_int[mask]
tm = t_int[mask]

f_int = np.zeros_like(t_int)
f_int[mask] = (xm**3 / np.expm1(xm)) / ((1.0 - tm)**2)
f_t[1:-1] = f_int

# Integración con Simpson 1/3 desde Fortran
I_num = mod.simpson(f_t, ht)
I_teo = np.pi**4 / 15.0
err_I = abs(I_num - I_teo)

# Constante de Stefan-Boltzmann
sigma_num = (2.0 * np.pi * k_B**4 * I_num) / (h_planck**3 * c**2)
sigma_teo = 5.670374419e-8

print(f"I numérico  = {I_num:.15f}")
print(f"I teórico   = {I_teo:.15f}")
print(f"Error abs I = {err_I:.3e}  (¿< 1e-10?: {err_I < 1e-10})")
print(f"sigma num   = {sigma_num:.10e} W m^-2 K^-4")
print(f"sigma teo   = {sigma_teo:.10e} W m^-2 K^-4")

I numérico  = 6.493939402266821
I teórico   = 6.493939402266828
Error abs I = 7.105e-15  (¿< 1e-10?: True)
sigma num   = 5.6703744192e-08 W m^-2 K^-4
sigma teo   = 5.6703744190e-08 W m^-2 K^-4


Actividad 4 — Comparación de tiempos

In [42]:
import time
import numpy as np

# Respaldo por si mejor_tiempo no está cargado en memoria
if "mejor_tiempo" not in globals():
    def mejor_tiempo(func, *args, repeticiones=5):
        tiempos = []
        for _ in range(repeticiones):
            t0 = time.perf_counter()
            func(*args)
            tiempos.append(time.perf_counter() - t0)
        return min(tiempos)

def trapecio_python(y, h):
    suma = 0.5 * (y[0] + y[-1])
    for i in range(1, len(y) - 1):
        suma += y[i]
    return h * suma

n = 10**6 + 1
x = np.linspace(0.0, np.pi, n)
h = x[1] - x[0]
y = np.sin(x)

# Compatibilidad con versiones de NumPy (<2.0 usa trapz, >=2.0 usa trapezoid)
np_trap = getattr(np, "trapezoid", getattr(np, "trapz", None))

t_py = mejor_tiempo(trapecio_python, y, h)
t_np = mejor_tiempo(np_trap, y, None, h)
t_f90 = mejor_tiempo(mod.trapecio, y, h)

factor = t_py / t_f90

print(f"Tiempo Python puro (for) : {t_py*1e3:10.4f} ms")
print(f"Tiempo NumPy (trapezoid) : {t_np*1e3:10.4f} ms")
print(f"Tiempo Fortran (F2PY)    : {t_f90*1e3:10.4f} ms")
print(f"Aceleración Python/F90   : {factor:10.2f}x más rápido")

Tiempo Python puro (for) :   172.5836 ms
Tiempo NumPy (trapezoid) :     5.7291 ms
Tiempo Fortran (F2PY)    :     2.0428 ms
Aceleración Python/F90   :      84.48x más rápido


**Parte D. Preguntas de interpretación**

1. **Orden de convergencia.** ¿Qué razones obtuvo en la actividad 2? ¿Qué relación tienen con el orden de cada regla?

Rta: En la actividad 2, al duplicar el número de subintervalos en cada malla ($n = 11, 21, 41, 81$, lo que equivale a reducir el paso $h$ a la mitad), las razones entre errores consecutivos $E(h)/E(h/2)$ obtenidas fueron $4.0049$, $4.0012$ y $4.0003$ para la regla del trapecio, y $16.1424$, $16.0353$ y $16.0088$ para la regla de Simpson 1/3. Esto refleja directamente el orden de convergencia $p$ de cada método: el error global de truncamiento escala como $E(h) \propto h^p$, de modo que al dividir el paso $h$ entre $2$ el error cae por un factor de $2^p$. Como el trapecio aproxima la curva mediante segmentos rectos, su error es cuadrático ($\mathcal{O}(h^2)$, $p = 2$) y se reduce por un factor de $2^2 = 4$; en cambio, Simpson ajusta arcos parabólicos que capturan la curvatura local, alcanzando un orden cuarto ($\mathcal{O}(h^4)$, $p = 4$) que hace disminuir el error por un factor de $2^4 = 16$ con los mismos nodos.

2. **Tiempos.** ¿Cuántas veces más rápido es el trapecio de Fortran que el de Python puro? ¿Por qué `np.trapezoid` también es mucho más rápido que el ciclo de Python?

Rta: Para una malla de $n = 10^6 + 1$ puntos, la subrutina compilada en Fortran tardó $2.0428\text{ ms}$ frente a los $172.5836\text{ ms}$ del ciclo `for` en Python puro, resultando **$84.48$ veces más rápida**. La diferencia ocurre porque en un bucle nativo de Python el intérprete debe verificar dinámicamente el tipo de dato, gestionar objetos en memoria y traducir instrucciones en cada una de las un millón de iteraciones, generando una enorme sobrecarga. Por su parte, `np.trapezoid` ($5.7291\text{ ms}$) —al igual que la subrutina de Fortran enlazada mediante F2PY— evita por completo esa interpretación paso a paso al ejecutar la suma directamente en código máquina compilado sobre un bloque contiguo y homogéneo de memoria (`float64`), permitiendo que el procesador opere los datos en bloque a máxima velocidad.

3. **Cuándo vale la pena.** Dé un ejemplo de un cálculo del curso que no se pueda escribir como una operación de NumPy sobre arreglos completos y explique por qué allí conviene una librería de Fortran.

Rta: Un ejemplo directo del curso es la integración temporal paso a paso de un sistema de ecuaciones diferenciales acopladas mediante Runge-Kutta de cuarto orden (RK4), como al simular la respuesta sísmica dinámica de un edificio de varios pisos, o una caminata aleatoria donde cada posición parte de la anterior. Estos algoritmos tienen una dependencia secuencial estricta entre iteraciones: no es posible vectorizarlos como una única operación simultánea sobre un arreglo completo de NumPy porque el estado en el instante $t_{k+1}$ no existe hasta haber calculado el estado en $t_k$. Si dejamos un ciclo de miles o millones de pasos temporales en Python puro, la sobrecarga del intérprete vuelve lenta la simulación; al encapsular ese núcleo secuencial en Fortran mediante F2PY, obtenemos velocidad nativa de código compilado en el bucle crítico sin renunciar a la comodidad de Python para preparar los parámetros y graficar los resultados.

**Parte E. Verificación**

In [43]:
# ==================== VERIFICACIÓN DEL TALLER 10 (no modificar) ====================
import os, sys, glob
import numpy as np

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ""))

# La verificación siempre trabaja dentro de la carpeta del taller
CARPETA = "taller_f2py"
if os.path.basename(os.getcwd()) != CARPETA:
    if not os.path.isdir(CARPETA):
        raise SystemExit(f"No existe la carpeta {CARPETA}: ejecute la celda de la Parte B.")
    os.chdir(CARPETA)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Verificando en:", os.getcwd(), "\n")

def libreria():
    if not os.path.isfile("modulo_integracion.f90"):
        raise ValueError(f"no existe modulo_integracion.f90 en {CARPETA}")
    so = glob.glob("integ.*.so") + glob.glob("integ.*.pyd")
    if not so:
        raise ValueError(f"no se encontró la librería integ en {CARPETA}; compile con F2PY")
    so = max(so, key=os.path.getmtime)                # la más reciente, si hay varias
    if os.path.getmtime(so) < os.path.getmtime("modulo_integracion.f90"):
        raise ValueError("la librería es anterior a modulo_integracion.f90; recompile y reinicie")
    import integ
    if os.path.abspath(integ.__file__) != os.path.abspath(so):
        raise ValueError(f"la librería cargada es {integ.__file__}, no la de {CARPETA}; reinicie el entorno")
    if not hasattr(integ, "integracion"):
        raise ValueError("la librería no contiene el módulo integracion")
    return integ.integracion

def muestras(f, a, b, n):
    x = np.linspace(a, b, n)
    return f(x), x[1] - x[0]

def c1():
    lib = libreria()
    y, h = muestras(lambda x: 3*x + 1, 0.0, 2.0, 7)        # recta: el trapecio es exacto
    if abs(lib.trapecio(y, h) - 8.0) > 1e-12:
        raise ValueError("trapecio no es exacto para una recta")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h*(0.5*(y[0] + y[-1]) + y[1:-1].sum())
    if abs(lib.trapecio(y, h) - ref) > 1e-13:
        raise ValueError("trapecio no coincide con la regla compuesta")
    return "exacto para una recta y correcto con sin x"

def c2():
    lib = libreria()
    y, h = muestras(lambda x: x**3 - 2*x, 0.0, 2.0, 5)     # cúbica: Simpson es exacto
    if abs(lib.simpson(y, h) - 0.0) > 1e-12:
        raise ValueError("simpson no es exacto para una cúbica")
    y, h = muestras(np.sin, 0.0, np.pi, 21)
    ref = h/3*(y[0] + y[-1] + 4*y[1:-1:2].sum() + 2*y[2:-1:2].sum())
    if abs(lib.simpson(y, h) - ref) > 1e-13:
        raise ValueError("simpson no coincide con la regla compuesta")
    return "exacto para una cúbica y correcto con sin x"

def c3():
    lib = libreria()
    eT, eS = [], []
    for n in [11, 21, 41, 81]:
        y, h = muestras(np.sin, 0.0, np.pi, n)
        eT.append(abs(lib.trapecio(y, h) - 2)); eS.append(abs(lib.simpson(y, h) - 2))
    rT = np.array(eT[:-1])/np.array(eT[1:]); rS = np.array(eS[:-1])/np.array(eS[1:])
    if np.any(abs(rT - 4) > 0.1) or np.any(abs(rS - 16) > 0.5):
        raise ValueError("las razones de convergencia no corresponden a órdenes 2 y 4")
    t = np.linspace(0.0, 1.0, 2001)
    f = np.zeros_like(t)
    x = t[1:-1]/(1 - t[1:-1])
    f[1:-1] = np.where(x < 700, x**3/np.expm1(np.minimum(x, 700))/(1 - t[1:-1])**2, 0.0)
    I = lib.simpson(f, t[1] - t[0])
    if abs(I - np.pi**4/15) > 1e-10:
        raise ValueError("la integral de Planck no alcanza la precisión pedida")
    return f"razones T ≈ {rT.mean():.2f}, S ≈ {rS.mean():.1f}; I = {I:.10f}"

caso("1. Regla del trapecio", c1)
caso("2. Regla de Simpson", c2)
caso("3. Orden de convergencia e integral de Planck", c3)
print(f"\nResultado: {sum(resultados)} de {len(resultados)} PASS")

Verificando en: /content/taller_cuerpo_negro/taller_f2py 

[PASS] 1. Regla del trapecio -> exacto para una recta y correcto con sin x
[PASS] 2. Regla de Simpson -> exacto para una cúbica y correcto con sin x
[PASS] 3. Orden de convergencia e integral de Planck -> razones T ≈ 4.00, S ≈ 16.1; I = 6.4939394023

Resultado: 3 de 3 PASS




---

